In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


In [2]:
# Code Evaluation for Max-of-K Circuit Analysis
# Repository: /net/scratch2/smallyan/max_of_k_claude_2025-12-26_01-10-45

REPO_PATH = "/net/scratch2/smallyan/max_of_k_claude_2025-12-26_01-10-45"

# List all files in the repository
print("Files in repository:")
for root, dirs, files in os.walk(REPO_PATH):
    # Skip hidden directories and evaluation directory
    dirs[:] = [d for d in dirs if not d.startswith('.') and d != 'evaluation']
    for f in files:
        if f.endswith(('.py', '.ipynb', '.md', '.txt')):
            rel_path = os.path.relpath(os.path.join(root, f), REPO_PATH)
            print(rel_path)

Files in repository:
logs/plan.md
logs/documentation.md
logs/code_walk.md
doc_only_evaluation/code_critic_evaluation.ipynb
doc_only_evaluation/replication_evaluation.md
doc_only_evaluation/self_matching.ipynb
doc_only_evaluation/generalization_eval.ipynb
notebooks/plan_Md.ipynb
notebooks/2025-12-26-01-10_CircuitAnalysis.ipynb
notebooks/code_walk_Md.ipynb
notebooks/documentation_Md.ipynb


In [3]:
# Read the plan file
with open(os.path.join(REPO_PATH, "logs/plan.md"), "r") as f:
    plan_content = f.read()
print("=== PLAN FILE ===")
print(plan_content)

=== PLAN FILE ===
# Max-of-K Circuit Analysis Plan

## Goal

Reverse-engineer the max-finding circuit in a 1-layer attention-only transformer to understand how the Query-Key (QK) attention mechanism implements the `argmax` operation for identifying the maximum value in a sequence.

## Hypothesis

The max-finding circuit uses **attention scores as value comparisons**:

1. **Embedding Layer**: Integer values are encoded such that embedding magnitude correlates with integer value.

2. **Attention Head (a0.h0)**:
   - **Query vectors**: Represent "looking for maximum"
   - **Key vectors**: Represent "this is my value" with magnitude proportional to integer value
   - **Attention score = Q · K**: Higher for larger values
   - The position with the highest value receives the highest attention weight

3. **Value Vectors & Output**:
   - Value vectors encode integer identity
   - Weighted sum via attention selects the maximum value's representation
   - Unembedding produces the maximum integer

In [4]:
# Read the codewalk file
with open(os.path.join(REPO_PATH, "logs/code_walk.md"), "r") as f:
    codewalk_content = f.read()
print("=== CODEWALK FILE ===")
print(codewalk_content)

=== CODEWALK FILE ===
# Max-of-K Circuit Analysis - Code Walkthrough

This document provides a detailed walkthrough of the code used to analyze the max-finding circuit in a 1-layer attention-only transformer.

## Table of Contents
1. [Environment Setup](#1-environment-setup)
2. [Dataset Generation](#2-dataset-generation)
3. [Model Architecture](#3-model-architecture)
4. [Training](#4-training)
5. [Circuit Analysis](#5-circuit-analysis)
6. [Visualizations](#6-visualizations)

---

## 1. Environment Setup

```python
import os
os.chdir('/home/smallyan/eval_agent')

import torch
import numpy as np
import random

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
```

**Purpose**: Set up the working directory, check for GPU availability, and set random seeds for rep

In [5]:
# Read the main implementation notebook
import json

notebook_path = os.path.join(REPO_PATH, "notebooks/2025-12-26-01-10_CircuitAnalysis.ipynb")
with open(notebook_path, "r") as f:
    notebook = json.load(f)

# Count cells
code_cells = [cell for cell in notebook['cells'] if cell['cell_type'] == 'code']
markdown_cells = [cell for cell in notebook['cells'] if cell['cell_type'] == 'markdown']
print(f"Total cells: {len(notebook['cells'])}")
print(f"Code cells: {len(code_cells)}")
print(f"Markdown cells: {len(markdown_cells)}")

Total cells: 21
Code cells: 15
Markdown cells: 6


In [6]:
# Print all code cells with their indices
for i, cell in enumerate(notebook['cells']):
    if cell['cell_type'] == 'code':
        source = ''.join(cell['source'])
        # Get first 100 chars or first line
        preview = source.strip()[:150].replace('\n', ' ')
        print(f"Cell {i}: {preview}...")

Cell 0: import os os.chdir('/home/smallyan/eval_agent') print(f"Working directory: {os.getcwd()}")...
Cell 1: # Check environment and GPU availability import torch import numpy as np import random  # Check GPU device = torch.device('cuda' if torch.cuda.is_avai...
Cell 3: # Generate max-finding dataset def generate_max_dataset(num_examples=10000, seq_len_range=(5, 10), value_range=(0, 99)):     """Generate dataset for m...
Cell 4: # Create PyTorch Dataset and DataLoader from torch.utils.data import Dataset, DataLoader  class MaxFindingDataset(Dataset):     """Dataset for max-fin...
Cell 6: import torch.nn as nn import torch.nn.functional as F import math  class AttentionOnlyTransformer(nn.Module):     """1-layer attention-only transforme...
Cell 7: # Training loop from tqdm import tqdm  def train_epoch(model, loader, optimizer, device):     model.train()     total_loss = 0     correct = 0     tot...
Cell 9: # Load best model model.load_state_dict(torch.load('best_model.pt')) model.eval(

In [7]:
# Setup for evaluation - inherit bashrc for model caching
import subprocess

# Get environment from bashrc
result = subprocess.run(
    ['bash', '-c', 'source ~/.bashrc && env'],
    capture_output=True, text=True
)

for line in result.stdout.split('\n'):
    if '=' in line and not line.startswith('_'):
        key, _, value = line.partition('=')
        if key in ['HF_HOME', 'TRANSFORMERS_CACHE', 'HF_HUB_CACHE', 'TORCH_HOME']:
            os.environ[key] = value
            print(f"Set {key}={value}")

# Set model cache directory
os.environ['HF_HUB_CACHE'] = '/net/projects2/chai-lab/shared_models'
print(f"\nHF_HUB_CACHE: {os.environ.get('HF_HUB_CACHE', 'not set')}")

Set HF_HOME=/net/projects2/chai-lab/shared_models

HF_HUB_CACHE: /net/projects2/chai-lab/shared_models


In [8]:
# Initialize evaluation tracking
evaluation_results = []

# Block 0 - Environment setup
block_id = "Cell_0"
block_desc = "Working directory setup"

print(f"=== Evaluating {block_id}: {block_desc} ===")

# Execute cell 0 code
cell_0_code = ''.join(notebook['cells'][0]['source'])
print("Code:")
print(cell_0_code)
print("\nExecution:")

=== Evaluating Cell_0: Working directory setup ===
Code:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Execution:


In [9]:
# Cell 0 already executed - it's simple setup, mark as passed
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

evaluation_results.append({
    "block_id": "Cell_0",
    "description": "Working directory setup",
    "runnable": "Y",
    "correct_implementation": "Y",
    "redundant": "N",
    "irrelevant": "N",
    "error_note": ""
})
print("\n✓ Cell 0 evaluation: PASSED")

Working directory: /home/smallyan/eval_agent

✓ Cell 0 evaluation: PASSED
